In [ ]:
import pandas as pd
import numpy as np

In [ ]:
index_2024

In [ ]:
index_2024['日期'] = pd.to_datetime(
    index_2024['日期'],
    format='%Y-%m-%d',    # 如果你的字符串里没有时分秒的话
    errors='coerce'       # 解析失败的会变成 NaT，方便你排查
)

In [ ]:
proportion_2024['日期'] = pd.to_datetime(
    proportion_2024['日期'],
    format='%Y-%m-%d',    # 如果你的字符串里没有时分秒的话
    errors='coerce'       # 解析失败的会变成 NaT，方便你排查
)

In [ ]:
import pandas as pd

# 假设两个 DataFrame 分别是 proportion_2024 和 index_2024，
# 且它们的“城市”和“日期”列都已是 datetime 格式

# 1. 只保留 index_2024 中你要的“三列”──城市、日期 和 迁徙指数
tmp = index_2024[['城市', '日期', '迁徙指数']].copy()

# 2. 跟 proportion_2024 左连接
merged = pd.merge(
    proportion_2024,
    tmp,
    on=['城市', '日期'],
    how='left'
)

# —— 2. 指定要填充的缺失城市 —— #
missing_cities = [
    '黔西南布依族苗族自治州',
    '黔东南苗族侗族自治州',
    '黔南布依族苗族自治州'
]
# —— 3. 从 index_2024 中取出每个城市 2024-01-23 和 2024-12-31 的“迁徙指数”作为随机边界 —— #
bounds = {}
for city in missing_cities:
    lo = index_2024.loc[
        (index_2024['城市'] == city) & (index_2024['日期'] == pd.to_datetime('2024-01-23')),
        '迁徙指数'
    ]
    hi = index_2024.loc[
        (index_2024['城市'] == city) & (index_2024['日期'] == pd.to_datetime('2024-12-31')),
        '迁徙指数'
    ]
    if not lo.empty and not hi.empty:
        bounds[city] = (lo.iloc[0], hi.iloc[0])
    else:
        # 如果没找到，就给出提醒
        print(f"Warning: 城市 {city} 缺少边界值，无法填充！")

# —— 4. 对每个城市的缺失行，用 np.random.uniform 在边界区间里随机抽样 —— #
np.random.seed(42)  # 如果你想要可复现的“随机”结果
for city, (low, high) in bounds.items():
    mask = (merged['城市'] == city) & (merged['迁徙指数'].isna())
    cnt = mask.sum()
    if cnt:
        merged.loc[mask, '迁徙指数'] = np.random.uniform(low, high, size=cnt)

# —— 5. 验证一下还有没有剩余的 NaN —— #
print("填充后剩余缺失：", merged['迁徙指数'].isna().sum())

In [ ]:
merged

In [ ]:
import pandas as pd

# 假设你的 DataFrame 是 merged

# 1. 只保留需要的列
df2 = merged[['日期', '城市', '目的城市', '比例', '迁徙指数']].copy()

# 2. 确保 '日期' 列是 datetime 类型
df2['日期'] = pd.to_datetime(df2['日期'])

# 3. 用 .dt.dayofyear 计算这一年中的第几天
df2['day_index'] = df2['日期'].dt.dayofyear

# 查看结果
print(df2.head())


In [ ]:
df2.to_csv('index2prop.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
data = pd.read_csv(r'C:\Users\Fe6ruar9\PhD\313.5-Heatwave\2024\index2prop_2024.csv', index_col=None, encoding='utf-8-sig')

In [ ]:
data.sort_values(by=['比例'], ascending=False)

In [ ]:
data['population'] = data['比例'] * data['迁徙指数'] * 32003.726712073505

In [ ]:
data.head()

In [ ]:
data.to_csv(r'H:\2024\to_mengqi.csv', index=False, encoding='utf-8-sig')